# Modelado — línea base y esquema de validación

Fase 5 de la guía orientativa. Este notebook fija dos cosas que no dependen de qué
features finales entregue `feature/preprocessing`: **la métrica de evaluación** y
**el esquema de validación**. Se apoya en `data/processed/train.csv` y `test.csv`
(salida de `transform.ipynb`, ya en `develop`), que solo trae las columnas de
calendario básicas.

Cuando `feature/preprocessing` cierre `train_features.csv` / `test_features.csv`
(con las variables de `feature_engineering.ipynb`: `es_festivo`, `tramo_tarde`,
`temporada`...), la comparativa de modelos (Paso 27) se repetirá sobre ese dataset
reutilizando la misma métrica y el mismo esquema de CV definidos aquí.

In [1]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)

## 1. Carga de datos

Una fila = un tramo horario (mañana/tarde) de un día concreto. El target es
`n_citas`: número de citas ejecutadas en ese tramo.

In [2]:
train = pd.read_csv('data/processed/train.csv', parse_dates=['fecha_cita'])
test = pd.read_csv('data/processed/test.csv', parse_dates=['fecha_cita'])

train = train.sort_values(['fecha_cita', 'tramo']).reset_index(drop=True)
test = test.sort_values(['fecha_cita', 'tramo']).reset_index(drop=True)

print(f"Train: {train.shape[0]} filas | {train['fecha_cita'].min().date()} -> {train['fecha_cita'].max().date()}")
print(f"Test:  {test.shape[0]} filas | {test['fecha_cita'].min().date()} -> {test['fecha_cita'].max().date()}")
train.head()

Train: 1252 filas | 2024-05-09 -> 2026-01-24
Test:  314 filas | 2026-01-25 -> 2026-06-30


,fecha_cita,tramo,n_citas,dia_semana,nombre_dia,es_finde,mes,anio,semana_iso
0,2024-05-09,mañana,0,3,Thursday,False,5,2024,19
1,2024-05-09,tarde,3,3,Thursday,False,5,2024,19
2,2024-05-10,mañana,1,4,Friday,False,5,2024,19
3,2024-05-10,tarde,2,4,Friday,False,5,2024,19
4,2024-05-11,mañana,2,5,Saturday,True,5,2024,19


## 2. Métrica de evaluación (Paso 25)

- **MAE (principal):** el error se interpreta directamente en "citas de más o de
  menos por tramo" — es la unidad que le sirve al negocio para decidir personal
  en sala. Robusta a algún día puntual con pico de demanda.
- **RMSE (secundaria):** penaliza más los errores grandes; interesa porque un
  fallo grande en un tramo (infra-dotar de personal un sábado tarde) cuesta más
  que varios fallos pequeños repartidos.
- **R² (referencia):** solo para contextualizar cuánta varianza explica el
  modelo frente al target; no se usa para decidir entre modelos porque no tiene
  unidades de negocio.

In [3]:
def evaluar(y_true, y_pred, nombre=''):
    resultado = {
        'modelo': nombre,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }
    return resultado

## 3. Modelos baseline (Paso 26)

Dos referencias mínimas que cualquier modelo con features debe superar:

1. **Media global** (`DummyRegressor`): predice siempre la media de `n_citas` en train.
2. **Estacional ingenuo (t-7)**: predice el valor observado en el mismo tramo
   siete días antes. No usa ninguna feature engineered, solo la estructura
   semanal ya confirmada en el EDA (§3.2: la demanda salta por día de la
   semana, no en gradiente).

In [4]:
# Baseline 1: media global
dummy = DummyRegressor(strategy='mean')
dummy.fit(train[['dia_semana']], train['n_citas'])  # el input no importa, solo predice la media
pred_dummy = dummy.predict(test[['dia_semana']])

In [5]:
# Baseline 2: estacional ingenuo (t-7), calculado sobre la serie completa
# ordenada por tramo para no mezclar mañana/tarde al desplazar
completo = pd.concat([train, test], ignore_index=True).sort_values(['tramo', 'fecha_cita'])
completo['pred_naive_t7'] = completo.groupby('tramo')['n_citas'].shift(7)

test_naive = completo[completo['fecha_cita'].isin(test['fecha_cita'])].dropna(subset=['pred_naive_t7'])
test_naive = test_naive.merge(test[['fecha_cita', 'tramo', 'n_citas']], on=['fecha_cita', 'tramo'], suffixes=('', '_test'))
print(f"Filas de test evaluables con t-7: {len(test_naive)} de {len(test)}")

Filas de test evaluables con t-7: 314 de 314


In [6]:
resultados = [
    evaluar(test['n_citas'], pred_dummy, 'Media global (dummy)'),
    evaluar(test_naive['n_citas'], test_naive['pred_naive_t7'], 'Estacional ingenuo (t-7)'),
]
tabla_baseline = pd.DataFrame(resultados).set_index('modelo').round(2)
tabla_baseline

,MAE,RMSE,R2
modelo,,,
Media global (dummy),2.77,3.71,-0.23
Estacional ingenuo (t-7),2.46,3.16,0.11


**Lectura:** el estacional ingenuo ya debería batir a la media global si la
estructura semanal pesa tanto como muestra el EDA. Si no lo hace, es señal de
alerta antes de invertir tiempo en modelos más complejos.

## 4. Esquema de validación cruzada (Paso 29-30)

**No usamos `KFold` aleatorio.** Mezclar fechas al azar entre folds dejaría que
un modelo "vea" información del futuro para predecir el pasado (fuga temporal),
lo cual infla artificialmente el rendimiento en CV y no se sostiene en
producción. Usamos `TimeSeriesSplit`: cada fold de validación queda siempre
*después* de su fold de entrenamiento.

In [7]:
tscv = TimeSeriesSplit(n_splits=5)
fechas = train['fecha_cita']

for i, (idx_tr, idx_val) in enumerate(tscv.split(train), start=1):
    print(f"Fold {i}: train hasta {fechas.iloc[idx_tr].max().date()} "
          f"({len(idx_tr)} filas) -> valida {fechas.iloc[idx_val].min().date()} "
          f"a {fechas.iloc[idx_val].max().date()} ({len(idx_val)} filas)")

Fold 1: train hasta 2024-08-22 (212 filas) -> valida 2024-08-23 a 2024-12-04 (208 filas)
Fold 2: train hasta 2024-12-04 (420 filas) -> valida 2024-12-05 a 2025-03-18 (208 filas)
Fold 3: train hasta 2025-03-18 (628 filas) -> valida 2025-03-19 a 2025-06-30 (208 filas)
Fold 4: train hasta 2025-06-30 (836 filas) -> valida 2025-07-01 a 2025-10-12 (208 filas)
Fold 5: train hasta 2025-10-12 (1044 filas) -> valida 2025-10-13 a 2026-01-24 (208 filas)


## 5. Próximos pasos

- Sustituir `train.csv`/`test.csv` por `train_features.csv`/`test_features.csv`
  en cuanto `feature/preprocessing` los cierre.
- Comparativa de modelos (Paso 27): `Ridge`, `RandomForestRegressor`,
  `GradientBoostingRegressor`, usando `tscv` y `evaluar()` ya definidos aquí.
- Optimización de hiperparámetros (Paso 28-31) sobre el/los 2-3 mejores.
- El test **no se ha tocado para elegir modelo** en este notebook — los
  baseline se evalúan contra test solo para fijar la cota mínima a superar, no
  para seleccionar hiperparámetros.